Disable PyTorch's strict safety checks entirely

In [1]:
import os
# Critical: Setting this BEFORE importing torch/ultralytics
os.environ['TORCH_FORCE_WEIGHTS_ONLY_LOAD'] = '0'

Cleanup Section

In [2]:
import shutil
from pathlib import Path
# Clean up previous runs for a fresh start
if Path("runs/detect/train").exists():
    shutil.rmtree("runs/detect/train", ignore_errors=True)
if Path("../data/yolo_dataset").exists():
    shutil.rmtree("../data/yolo_dataset", ignore_errors=True)
print("Setup complete. Previous runs cleaned up.")

Setup complete. Previous runs cleaned up.


Setup and Imports

In [3]:
from pathlib import Path

# Install necessary libraries
%pip install ultralytics==8.0.196 scikit-learn datasets pyyaml opencv-python pillow matplotlib

# Import modules
from lucky_data_prep import load_and_merge_datasets, MaskToBBoxConverter 
from lucky_yolo_trainer import CustomYOLOTrainer

# --- IMPROVED Configuration ---
DATA_YAML_PATH = "../data/yolo_dataset/data.yaml"

# Used nano model
MODEL_WEIGHTS = "yolov8n.pt"

Note: you may need to restart the kernel to use updated packages.


Step 1 - Load and Merge Data

In [4]:
dataset = load_and_merge_datasets()

if dataset is None:
    print("Pipeline stopped: Failed to load dataset.")


STEP 1: LOADING BUSI DATASET
Checking for RAW data at: /home/teaching/ADL-project/data/Dataset_BUSI_with_GT

Total BUSI images loaded: 780

Loading HuggingFace dataset (nielsr/breast-cancer)...
Loaded 130 images from HuggingFace
Converted 130 HuggingFace images

Successfully merged datasets!
  - BUSI: 780 images
  - HuggingFace: 130 images
  - Total: 910 images


Step 2 - Convert Data to YOLO Format

In [5]:
if dataset:
    converter = MaskToBBoxConverter()
    stats = converter.convert_dataset(dataset)
    
    if stats:
        print("\nDataset ready for YOLO training.")
    else:
        print("Pipeline stopped: Failed to convert dataset.")


STEP 2: CONVERTING TO YOLO FORMAT
Created YOLO directory structure
Total Records: 910
Split sizes: Train=637, Val=136, Test=137

Processing train split...

Processing val split...

Processing test split...

Conversion complete:
  train: 637 images
  val: 136 images
  test: 137 images
Created data.yaml at ../data/yolo_dataset/data.yaml

Dataset ready for YOLO training.


Diagnostics

In [6]:
import os
import yaml

# Check the path set in Cell 1
DATA_YAML_PATH = "../data/yolo_dataset/data.yaml"

if os.path.exists(DATA_YAML_PATH):
    print(f"--- Content of {DATA_YAML_PATH} ---")
    with open(DATA_YAML_PATH, 'r') as f:
        data_content = yaml.safe_load(f)
        print(yaml.dump(data_content, indent=4))
    
    # Check if the images/labels exist relative to the 'path'
    base_path = data_content.get('path', 'NOT_FOUND')
    train_images = os.path.join(base_path, data_content.get('train', 'images/train'))
    
    print(f"\n--- Checking Internal Paths ---")
    print(f"Base Path: {base_path}")
    print(f"Train Images Path: {train_images}")
    
    if os.path.exists(train_images):
        print(f"Train images directory found.")
        print(f"Number of files in train/images: {len(os.listdir(train_images))}")
    else:
        print(f"Train images directory NOT FOUND. Check your 'path' in data.yaml.")

else:
    print(f"ERROR: data.yaml not found at {DATA_YAML_PATH}")

--- Content of ../data/yolo_dataset/data.yaml ---
names:
- tumor
nc: 1
path: /home/teaching/ADL-project/notebooks/../data/yolo_dataset
test: images/test
train: images/train
val: images/val


--- Checking Internal Paths ---
Base Path: /home/teaching/ADL-project/notebooks/../data/yolo_dataset
Train Images Path: /home/teaching/ADL-project/notebooks/../data/yolo_dataset/images/train
Train images directory found.
Number of files in train/images: 637


Step 3 - Load Model and Freeze Layers

In [7]:
# Initialize trainer
trainer = CustomYOLOTrainer(model_size=MODEL_WEIGHTS, data_yaml=DATA_YAML_PATH)

# Load base model (uses the PyTorch fix)
if trainer.load_model():
    # Freeze backbone layers (a key step for better fine-tuning)
    trainer.freeze_backbone(num_layers=10)


STEP 3: LOADING PRE-TRAINED YOLO MODEL (yolov8n.pt)
PyTorch version: 2.8.0+cu128
Successfully loaded yolov8n.pt

FREEZING BACKBONE LAYERS (0 to 9)
Successfully froze 10 layers. Head remains trainable.


Step 4 - Fine-Tune the Model

In [8]:
if trainer.model:
    print("\n" + "=" * 60)
    print(f"STARTING FINE-TUNING on {MODEL_WEIGHTS} (50 epochs)")
    print("=" * 60)
    
    # **Run the training:**
    results = trainer.train(epochs=50, batch_size=8, patience=10) 
    
    if results:
        print(f"Training finished. Best model saved to: {results.save_dir}/weights/best.pt")
        # Save the path to the best model for the validation step
        best_weights_path = Path(results.save_dir) / "weights" / "best.pt"
    else:
        best_weights_path = None
        print("Training failed. Validation skipped.")
else:
    best_weights_path = None



STARTING FINE-TUNING on yolov8n.pt (50 epochs)

STEP 4: TRAINING YOLO MODEL
Training on: cuda


New https://pypi.org/project/ultralytics/8.3.221 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.0.196 🚀 Python-3.13.5 torch-2.8.0+cu128 CUDA:0 (NVIDIA RTX A5000, 24135MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=../data/yolo_dataset/data.yaml, epochs=50, patience=10, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=cuda, workers=8, project=None, name=None, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, show=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, vid_stride=1, stream_buffer=False, line_width=None, visualize=False, augment


Training complete!
Training finished. Best model saved to: /home/teaching/ADL-project/runs/detect/train/weights/best.pt


Step 5 - Final Validation

In [9]:
if best_weights_path and best_weights_path.exists(): 
    print(f"\n" + "=" * 60)
    print("STEP 5: LOADING BEST MODEL AND VALIDATING")
    print("=" * 60)
    
    # Load the best weights from the training run
    final_trainer = CustomYOLOTrainer(model_size=str(best_weights_path), data_yaml=DATA_YAML_PATH)
    final_trainer.load_model()
    
    # **Run final validation:**
    if final_trainer.model:
        metrics = final_trainer.validate()
        
        if metrics:
            print("\nFinal Model Performance is in the metrics above.")
else:
    print("\nFinal Validation Skipped: No best model weights found.")


Ultralytics YOLOv8.0.196 🚀 Python-3.13.5 torch-2.8.0+cu128 CUDA:0 (NVIDIA RTX A5000, 24135MiB)
Model summary (fused): 168 layers, 3005843 parameters, 0 gradients, 8.1 GFLOPs



STEP 5: LOADING BEST MODEL AND VALIDATING

STEP 3: LOADING PRE-TRAINED YOLO MODEL (/home/teaching/ADL-project/runs/detect/train/weights/best.pt)
PyTorch version: 2.8.0+cu128
Successfully loaded /home/teaching/ADL-project/runs/detect/train/weights/best.pt

STEP 5: VALIDATING DETECTION ACCURACY


val: Scanning /home/teaching/ADL-project/data/yolo_dataset/labels/test... 137 images, 22 backgrounds, 0 corrupt: 100%|██████████| 137/137 [00:00<00:00, 1935.48it/s]
val: New cache created: /home/teaching/ADL-project/data/yolo_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:01<00:00,  7.05it/s]
                   all        137        115      0.862      0.813      0.871       0.57
Speed: 0.1ms preprocess, 3.6ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /home/teaching/ADL-project/runs/detect/val



Metrics:
 mAP50: 0.8714
 mAP50-95: 0.5696
 Precision: 0.8618
 Recall: 0.8132

Final Model Performance is in the metrics above.
